# MVP Test Cases

这个 notebook 专门用于回归测试和临时加 case。核心逻辑已经迁移到 `merchant_growth_mvp.py`。

In [ ]:
import os
from pathlib import Path

import pandas as pd

from merchant_growth_mvp import (
    GAP_CONFIG,
    HEALTH_CONFIG,
    create_mock_inputs,
    get_brand_row,
    load_local_env,
    parse_intent_payload,
    resolve_env_path,
    run_llm_orchestrated_pipeline,
    run_pipeline,
)

merchant_df, peer_benchmark = create_mock_inputs()
default_brand_id = "B001"
brand_row = get_brand_row(merchant_df, default_brand_id)

## API Key Check

先运行这一格，确认当前 notebook 读到的是哪个 `.env`，以及 `OPENAI_API_KEY` 有没有成功加载。读取优先级是：`MERCHANT_AGENT_ENV_PATH` -> `~/.merchant-agent/.env` -> 项目内 `.env`。

In [ ]:
configured_env_path = os.getenv("MERCHANT_AGENT_ENV_PATH")
resolved_env_path = resolve_env_path()
env_loaded = load_local_env()

print("MERCHANT_AGENT_ENV_PATH:", configured_env_path)
print("Resolved env path:", Path(resolved_env_path).expanduser())
print("Found env file:", Path(resolved_env_path).expanduser().exists())
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))


## Interactive Run

在下面这个单元修改 `interactive_brand_id` 和 `interactive_question`，然后运行下一格，就会完整跑一遍：意图识别 -> 分析模块 -> 输出结果。

In [ ]:
interactive_brand_id = "B001"
interactive_question = "为什么最近订单下降了？"
interactive_mock_mode = True

In [ ]:
interactive_result = run_llm_orchestrated_pipeline(
    brand_id=interactive_brand_id,
    question=interactive_question,
    merchant_df=merchant_df,
    peer_benchmark=peer_benchmark,
    health_config=HEALTH_CONFIG,
    gap_config=GAP_CONFIG,
    mock_mode=interactive_mock_mode,
)

print("=== Question ===")
print(interactive_question)
print()
print("=== Brand Scope ===")
print({"brand_id": interactive_result["brand_id"], "brand_name": interactive_result["brand_name"]})
print()
print("=== Intent Result ===")
print(interactive_result["intent_result"])
print()
if interactive_result.get("intercepted"):
    print("=== Intercepted ===")
    print("当前问题被识别为无关问题，未进入业务分析模块。")
else:
    print("=== Structured Output ===")
    interactive_result["structured_output"]


In [ ]:
print("=== Controlled LLM Output ===")
print(interactive_result["analysis_result"]["text"])
print()
print("=== Token Usage ===")
print("Estimated input tokens:", interactive_result["analysis_result"]["estimated_input_tokens"])
print("Mode:", interactive_result["analysis_result"]["mode"])


## Pipeline Cases

In [ ]:
test_cases = [
    {
        "name": "root_cause only returns health + gap",
        "question": "为什么最近订单下降了？",
        "expected_modules": ["health", "gap"],
        "present_keys": ["health", "gaps"],
        "absent_keys": ["actions"],
    },
    {
        "name": "action_recommendation returns gap + action",
        "question": "给我一些提升订单的建议",
        "expected_modules": ["gap", "action"],
        "present_keys": ["gaps", "actions"],
        "absent_keys": ["health"],
    },
    {
        "name": "default diagnosis returns all modules",
        "question": "帮我全面诊断一下这个商家",
        "expected_modules": ["health", "gap", "action"],
        "present_keys": ["health", "gaps", "actions"],
        "absent_keys": [],
    },
    {
        "name": "irrelevant question is intercepted",
        "question": "今天天气怎么样？",
        "expected_modules": [],
        "present_keys": [],
        "absent_keys": ["health", "gaps", "actions"],
    },
]

rows = []
for case in test_cases:
    brand_row = get_brand_row(merchant_df, default_brand_id)
    output = run_pipeline(
        merchant_row=brand_row,
        question=case["question"],
        peer_benchmark=peer_benchmark,
        health_config=HEALTH_CONFIG,
        gap_config=GAP_CONFIG,
    )

    assert output["plan"]["analysis_modules"] == case["expected_modules"]
    for key in case["present_keys"]:
        assert key in output
    for key in case["absent_keys"]:
        assert key not in output

    rows.append(
        {
            "case": case["name"],
            "brand_id": default_brand_id,
            "question": case["question"],
            "modules": ", ".join(output["plan"]["analysis_modules"]),
            "returned_keys": ", ".join(k for k in ["health", "gaps", "actions"] if k in output),
        }
    )

pd.DataFrame(rows)

## Intent Parsing Cases

In [ ]:
intent_parse_cases = [
    ('{"intent": "diagnosis", "reason": "用户要整体看经营情况"}', "diagnosis"),
    ('```json\n{"intent": "root_cause", "reason": "用户在追问下降原因"}\n```', "root_cause"),
    ('模型解释如下：{"intent": "action_recommendation", "reason": "用户想要建议"}', "action_recommendation"),
]

for raw_text, expected_intent in intent_parse_cases:
    parsed = parse_intent_payload(raw_text)
    assert parsed["intent"] == expected_intent

try:
    parse_intent_payload('{"intent": "unknown", "reason": "bad"}')
    raise AssertionError("非法 intent 应该触发异常")
except ValueError:
    pass

print("Intent JSON parsing tests passed.")

## LLM Orchestrated Smoke Test

In [ ]:
llm_result = run_llm_orchestrated_pipeline(
    brand_id="B001",
    question="给我一些提升订单的建议",
    merchant_df=merchant_df,
    peer_benchmark=peer_benchmark,
    health_config=HEALTH_CONFIG,
    gap_config=GAP_CONFIG,
    mock_mode=True,
)

print(llm_result["intent_result"])
print()
print(llm_result["analysis_result"]["text"])

## Irrelevant Question Smoke Test

In [ ]:
irrelevant_result = run_llm_orchestrated_pipeline(
    brand_id="B001",
    question="帮我推荐一部电影",
    merchant_df=merchant_df,
    peer_benchmark=peer_benchmark,
    health_config=HEALTH_CONFIG,
    gap_config=GAP_CONFIG,
    mock_mode=True,
)

assert irrelevant_result["intercepted"] is True
print(irrelevant_result["intent_result"])
print()
print(irrelevant_result["analysis_result"]["text"])
